# MedAgent-CDSS - Evaluation

**Educational simulation only.** This notebook runs the evaluation framework (`app/evaluation.py`) on the synthetic cases. The metrics describe *software behaviour* (validation, completion, structured-output validity, safety-rule coverage, latency). They are **not** measures of clinical accuracy.

Run it from the `notebooks/` folder with the project virtual environment (`pip install notebook` if Jupyter is not installed).

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.environ.setdefault("USE_LLM", "false")  # set to "true" (with an API key) to also evaluate LLM mode

In [ ]:
from IPython.display import Markdown, display

from app.evaluation import run_evaluation, to_markdown

results = run_evaluation(include_llm=os.environ["USE_LLM"] == "true", write_files=False)
display(Markdown(to_markdown(results)))

## Per-case results (rule-based mode)

In [ ]:
import pandas as pd

pd.DataFrame(results["modes"]["rule-based"]["per_case"])

## Mean latency per agent (ms)

In [ ]:
pd.Series(results["modes"]["rule-based"]["mean_agent_latency_ms"]).sort_values(ascending=False)

## Safety rule coverage

Measured on the small hand-written labelled set in `evaluation/safety_test_set.json`.

In [ ]:
safety = results["safety_rules"]
print("Detection rate:", safety["violation_detection_rate"])
print("False-positive rate:", safety["false_positive_rate"])
print("Missed unsafe sentences:", safety["missed"])
print("False positives:", safety["false_positives"])

## Interpretation and limitations

- The synthetic cases and the knowledge base were written for this project, so a high *intended-pattern* match mostly shows the pipeline is wired correctly - it does not show generalisation.
- The safety test set is small and hand-written; regex rules can miss paraphrased unsafe advice.
- LLM-mode metrics are only available when an API key is configured and will vary between runs and models.